In [1]:
!git clone https://github.com/VladWero08/time-series-ad-gan.git

fatal: destination path 'time-series-ad-gan' already exists and is not an empty directory.


In [2]:
pip install pandas numpy kagglehub torch scipy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys 
import os

module_path = os.path.abspath("./time-series-ad-gan")
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
import ast
import pandas as pd
import numpy as np
import json
import kagglehub
import typing as t
import torch
import matplotlib.pyplot as plt
from urllib.request import urlopen

from src.models.mad_gan import run_pipeline
from src.utils.data import intervals_to_points
from src.utils.errors import point_wise_error, area_wise_error, dtw_error

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Working on {device}!")

Working on cuda!


In [6]:
def plot_signal(X: np.ndarray, y: np.ndarray) -> None:
    plt.figure(figsize=(15, 4))
    for idx in np.where(y == 1)[0]:
        plt.axvline(idx, color='red', alpha=0.4, linewidth=0.8, zorder=0)
    plt.plot(X, zorder=1)
    plt.show()

## **Yahoo S5**

In [7]:
A1_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A1Benchmark/"
A1_FILE_NAME = "real_"
A1_N_FILES = 67

A2_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A2Benchmark/"
A2_FILE_NAME = "synthetic_"
A2_N_FILES = 100

A3_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A3Benchmark/"
A3_FILE_NAME = "A3Benchmark-TS"
A3_N_FILES = 100

A4_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A4Benchmark/"
A4_FILE_NAME = "A4Benchmark-TS"
A4_N_FILES = 100

In [ ]:
YAHOO_HYPERPARAMS = {
    "train_ratio": 0.6,
    "device": device,
    "anomaly_type": "contextual",
    "verbose": False,
    "lr_g": 1e-5,
    "lr_d": 1e-5,
    "epochs": 1000,
    "rec_error_funcs": [("point", point_wise_error), ("area", area_wise_error)],
}

In [9]:
def yahoo_download_subdataset(folder: str, file_name: str, n_files: int) -> t.List[pd.DataFrame]:
    yahoo_signals = []
    yahoo_points = 0
    yahoo_anomalies = 0

    for i in range(1, n_files + 1):
        # name of the .csv in the repository
        yahoo_fn = f"{file_name}{i}.csv"
        # url to the raw .csv file in the repository
        yahoo_url = f"{folder}{yahoo_fn}"
        yahoo_df = pd.read_csv(yahoo_url)
        yahoo_df = yahoo_df.rename(columns={"anomaly": "is_anomaly", "timestamps": "timestamp"})
        
        # count the number of points and the number of anomaly points
        yahoo_points += len(yahoo_df)
        yahoo_anomalies += (yahoo_df["is_anomaly"] == 1).sum()
        yahoo_signals.append(yahoo_df)

        if i % 10 == 0:
            print(f"Downloaded {i} .csv files.")

    print()
    print("Finished downloading!")
    print("---------------------")
    print(f"A1 Total Signals: {A1_N_FILES}")
    print(F"A1 Total Points: {yahoo_points}")
    print(f"A1 Total Anomaly Points: {yahoo_anomalies}")
    print(f"A1 Anomaly Rate: {(yahoo_anomalies / yahoo_points) * 100:.2f}%")

    return yahoo_signals

### **A1**

In [11]:
a1_signals = yahoo_download_subdataset(folder=A1_FOLDER, file_name=A1_FILE_NAME, n_files=A1_N_FILES)

Downloaded 10 .csv files.
Downloaded 20 .csv files.
Downloaded 30 .csv files.
Downloaded 40 .csv files.
Downloaded 50 .csv files.
Downloaded 60 .csv files.

Finished downloading!
---------------------
A1 Total Signals: 67
A1 Total Points: 94866
A1 Total Anomaly Points: 1669
A1 Anomaly Rate: 1.76%


In [12]:
total_metrics = {name: np.zeros(3) for (name, _) in YAHOO_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a1_signal in enumerate(a1_signals):
    X = a1_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a1_signal['is_anomaly'].to_numpy()

    print(f"A1 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **YAHOO_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A1 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

A1 Signal 1...


/venv/main/lib/python3.12/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


A1 Signal 2...
A1 Signal 3...
A1 Signal 4...
A1 Signal 5...
A1 Signal 6...
A1 Signal 7...
A1 Signal 8...
A1 Signal 9...
A1 Signal 10...
A1 Signal 11...
A1 Signal 12...
A1 Signal 13...
A1 Signal 14...
A1 Signal 15...
A1 Signal 16...
A1 Signal 17...
A1 Signal 18...
A1 Signal 19...
A1 Signal 20...
A1 Signal 21...
A1 Signal 22...
A1 Signal 23...
A1 Signal 24...
A1 Signal 25...
A1 Signal 26...
A1 Signal 27...
A1 Signal 28...
A1 Signal 29...
A1 Signal 30...
A1 Signal 31...
A1 Signal 32...
A1 Signal 33...
A1 Signal 34...
A1 Signal 35...
A1 Signal 36...
A1 Signal 37...
A1 Signal 38...
A1 Signal 39...
A1 Signal 40...
A1 Signal 41...
A1 Signal 42...
A1 Signal 43...
A1 Signal 44...
A1 Signal 45...
A1 Signal 46...
A1 Signal 47...
A1 Signal 48...
A1 Signal 49...
A1 Signal 50...
A1 Signal 51...
A1 Signal 52...
A1 Signal 53...
A1 Signal 54...
A1 Signal 55...
A1 Signal 56...
A1 Signal 57...
A1 Signal 58...
A1 Signal 59...
A1 Signal 60...
A1 Signal 61...
A1 Signal 62...
A1 Signal 63...
A1 Signal 64...


### **A2**

In [13]:
A2_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A2_HYPERPARAMS["train_ratio"] = 0.5

In [14]:
a2_signals = yahoo_download_subdataset(folder=A2_FOLDER, file_name=A2_FILE_NAME, n_files=A2_N_FILES)

Downloaded 10 .csv files.
Downloaded 20 .csv files.
Downloaded 30 .csv files.
Downloaded 40 .csv files.
Downloaded 50 .csv files.
Downloaded 60 .csv files.
Downloaded 70 .csv files.
Downloaded 80 .csv files.
Downloaded 90 .csv files.
Downloaded 100 .csv files.

Finished downloading!
---------------------
A1 Total Signals: 67
A1 Total Points: 142100
A1 Total Anomaly Points: 466
A1 Anomaly Rate: 0.33%


In [15]:
total_metrics = {name: np.zeros(3) for (name, _) in A2_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a2_signal in enumerate(a2_signals):
    X = a2_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a2_signal['is_anomaly'].to_numpy()
    
    print(f"A2 Signal {i + 1}...")
        
    try:
        metrics = run_pipeline(X, y, **A2_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A2 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

A2 Signal 1...


/venv/main/lib/python3.12/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


A2 Signal 2...
A2 Signal 3...
A2 Signal 4...
A2 Signal 5...
A2 Signal 6...
A2 Signal 7...
A2 Signal 8...
A2 Signal 9...
A2 Signal 10...
A2 Signal 11...
A2 Signal 12...
A2 Signal 13...
A2 Signal 14...
A2 Signal 15...
A2 Signal 16...
A2 Signal 17...
A2 Signal 18...
A2 Signal 19...
A2 Signal 20...
A2 Signal 21...
A2 Signal 22...
A2 Signal 23...
A2 Signal 24...
A2 Signal 25...
A2 Signal 26...
A2 Signal 27...
A2 Signal 28...
A2 Signal 29...
A2 Signal 30...
A2 Signal 31...
A2 Signal 32...
A2 Signal 33...
A2 Signal 34...
A2 Signal 35...
A2 Signal 36...
A2 Signal 37...
A2 Signal 38...
A2 Signal 39...
A2 Signal 40...
A2 Signal 41...
A2 Signal 42...
A2 Signal 43...
A2 Signal 44...
A2 Signal 45...
A2 Signal 46...
A2 Signal 47...
A2 Signal 48...
A2 Signal 49...
A2 Signal 50...
A2 Signal 51...
A2 Signal 52...
A2 Signal 53...
A2 Signal 54...
A2 Signal 55...
A2 Signal 56...
A2 Signal 57...
A2 Signal 58...
A2 Signal 59...
A2 Signal 60...
A2 Signal 61...
A2 Signal 62...
A2 Signal 63...
A2 Signal 64...


### **A3**

In [10]:
A3_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A3_HYPERPARAMS["train_ratio"] = 0.5

In [11]:
a3_signals = yahoo_download_subdataset(folder=A3_FOLDER, file_name=A3_FILE_NAME, n_files=A3_N_FILES)

Downloaded 10 .csv files.
Downloaded 20 .csv files.
Downloaded 30 .csv files.
Downloaded 40 .csv files.
Downloaded 50 .csv files.
Downloaded 60 .csv files.
Downloaded 70 .csv files.
Downloaded 80 .csv files.
Downloaded 90 .csv files.
Downloaded 100 .csv files.

Finished downloading!
---------------------
A1 Total Signals: 67
A1 Total Points: 168000
A1 Total Anomaly Points: 943
A1 Anomaly Rate: 0.56%


In [12]:
total_metrics = {name: np.zeros(3) for (name, _) in A3_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a3_signal in enumerate(a3_signals):
    X = a3_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a3_signal['is_anomaly'].to_numpy()
    
    print(f"A3 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **A3_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A3 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

A3 Signal 1...


/venv/main/lib/python3.12/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


A3 Signal 2...
A3 Signal 3...
A3 Signal 4...
A3 Signal 5...
A3 Signal 6...
A3 Signal 7...
A3 Signal 8...
A3 Signal 9...
A3 Signal 10...
A3 Signal 11...
A3 Signal 12...
A3 Signal 13...
A3 Signal 14...
A3 Signal 15...
A3 Signal 16...
A3 Signal 17...
A3 Signal 18...
A3 Signal 19...
A3 Signal 20...
A3 Signal 21...
A3 Signal 22...
A3 Signal 23...
A3 Signal 24...
A3 Signal 25...
A3 Signal 26...
A3 Signal 27...
A3 Signal 28...
A3 Signal 29...
A3 Signal 30...
A3 Signal 31...
A3 Signal 32...
A3 Signal 33...
A3 Signal 34...
A3 Signal 35...
A3 Signal 36...
A3 Signal 37...
A3 Signal 38...
A3 Signal 39...
A3 Signal 40...
A3 Signal 41...
A3 Signal 42...
A3 Signal 43...
A3 Signal 44...
A3 Signal 45...
A3 Signal 46...
A3 Signal 47...
A3 Signal 48...
A3 Signal 49...
A3 Signal 50...
A3 Signal 51...
A3 Signal 52...
A3 Signal 53...
A3 Signal 54...
A3 Signal 55...
A3 Signal 56...
A3 Signal 57...
A3 Signal 58...
A3 Signal 59...
A3 Signal 60...
A3 Signal 61...
A3 Signal 62...
A3 Signal 63...
A3 Signal 64...


### **A4**

In [15]:
A4_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A4_HYPERPARAMS["train_ratio"] = 0.5

In [16]:
a4_signals = yahoo_download_subdataset(folder=A4_FOLDER, file_name=A4_FILE_NAME, n_files=A4_N_FILES)

Downloaded 10 .csv files.
Downloaded 20 .csv files.
Downloaded 30 .csv files.
Downloaded 40 .csv files.
Downloaded 50 .csv files.
Downloaded 60 .csv files.
Downloaded 70 .csv files.
Downloaded 80 .csv files.
Downloaded 90 .csv files.
Downloaded 100 .csv files.

Finished downloading!
---------------------
A1 Total Signals: 67
A1 Total Points: 168000
A1 Total Anomaly Points: 837
A1 Anomaly Rate: 0.50%


In [17]:
total_metrics = {name: np.zeros(3) for (name, _) in A4_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a4_signal in enumerate(a4_signals):
    X = a4_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a4_signal['is_anomaly'].to_numpy()
    
    print(f"A4 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **A4_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A4 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

A4 Signal 1...
A4 Signal 2...
A4 Signal 3...
A4 Signal 4...
A4 Signal 5...
A4 Signal 6...
A4 Signal 7...
A4 Signal 8...
A4 Signal 9...
A4 Signal 10...
A4 Signal 11...
A4 Signal 12...
A4 Signal 13...
A4 Signal 14...
A4 Signal 15...
A4 Signal 16...
A4 Signal 17...
A4 Signal 18...
A4 Signal 19...
A4 Signal 20...
A4 Signal 21...
A4 Signal 22...
A4 Signal 23...
A4 Signal 24...
A4 Signal 25...
A4 Signal 26...
A4 Signal 27...
A4 Signal 28...
A4 Signal 29...
A4 Signal 30...
A4 Signal 31...
A4 Signal 32...
A4 Signal 33...
A4 Signal 34...
A4 Signal 35...
A4 Signal 36...
A4 Signal 37...
A4 Signal 38...
A4 Signal 39...
A4 Signal 40...
A4 Signal 41...
A4 Signal 42...
A4 Signal 43...
A4 Signal 44...
A4 Signal 45...
A4 Signal 46...
A4 Signal 47...
A4 Signal 48...
A4 Signal 49...
A4 Signal 50...
A4 Signal 51...
A4 Signal 52...
A4 Signal 53...
A4 Signal 54...
A4 Signal 55...
A4 Signal 56...
A4 Signal 57...
A4 Signal 58...
A4 Signal 59...
A4 Signal 60...
A4 Signal 61...
A4 Signal 62...
A4 Signal 63...
A